# MobileBERT SMS Transaction Classifier — TensorFlow Edition

Fine-tunes `google/mobilebert-uncased` on 100 K Indian bank SMS messages to classify **debit transactions** vs non-transactions.  
Exports a TFLite model (with BertNLClassifier-compatible metadata) directly into `app/src/main/assets/model.tflite`.

**Why TensorFlow instead of PyTorch + ai-edge-torch?**  
Native TF → TFLite conversion via `tf.lite.TFLiteConverter` is first-class and reliable.  
`ai-edge-torch` (PyTorch → TFLite) requires extra wrapper logic and can fail on transformer attention ops.

---
**Pipeline overview**
```
refined_training_data.csv
        │
        ▼
  EDA & Preprocessing
        │
        ▼
  MobileBERT fine-tune (TensorFlow / Keras)
        │
        ▼
  Evaluation  ──→  metrics / plots
        │
        ▼
  tf.lite.TFLiteConverter  ──→  model_unquantized.tflite
        │
        ▼
  TFLite metadata injection (BertNLClassifier)
        │
        ▼
  app/src/main/assets/model.tflite
```

**Python requirement:** ≥ 3.10  
**Virtual environment:** `tf_env/` (separate from `ml_env/`)

## 1  Environment Setup

**Local:** Creates an isolated `tf_env/` virtual environment pinned to **Python 3.11** (the highest  
version supported by `tflite-support 0.4.4`), installs all TF-stack dependencies, and registers a Jupyter kernel.

**Google Colab:** Auto-detected — packages are installed directly into the Colab runtime. No venv needed.

**No PyTorch, no ai-edge-torch** are installed in either path.

In [ ]:
import subprocess, sys, os
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules

# ── package list (same for both Colab and local) ──────────────────────────────
PACKAGES = [
    "wheel",
    "setuptools>=68",
    # TensorFlow core
    "tensorflow>=2.16.0",
    "tf-keras>=2.16.0",          # HuggingFace TF backend targets tf-keras, not Keras 3
    # HuggingFace (TF backend only — no torch, no ai-edge-torch)
    "transformers>=4.40.0",
    # TFLite metadata — Python 3.11 max; replaces manual flatbuffers builder
    "tflite-support>=0.4.4",
    # data / evaluation
    "pandas>=2.1.0",
    "numpy>=1.26.0",
    "scikit-learn>=1.4.0",
    "matplotlib>=3.8.0",
    "seaborn>=0.13.0",
    "tqdm>=4.66.0",
    # Jupyter
    "ipykernel",
    "ipywidgets",
]

if IS_COLAB:
    # ── Google Colab: install directly, no venv ───────────────────────────────
    print("Google Colab detected — installing packages into current Python environment …")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + PACKAGES)

    # Colab sometimes pre-installs an older TF; verify the version after install.
    import importlib
    import tensorflow as _tf_check
    if _tf_check.__version__ < "2.16":
        print(
            "\n[ACTION REQUIRED] TensorFlow version mismatch.\n"
            "Please restart the Colab runtime:  Runtime → Restart runtime\n"
            "Then re-run from Section 2 (skip this cell)."
        )
    else:
        print(f"TensorFlow {_tf_check.__version__} OK. Continue from Section 2.")

else:
    # ── Local: create isolated Python 3.11 venv ───────────────────────────────
    PY311    = "python3.11"
    VENV_DIR = Path(os.getcwd()) / "tf_env"
    VENV_PY  = VENV_DIR / "bin" / "python"
    VENV_PIP = VENV_DIR / "bin" / "pip"

    # Verify python3.11 is available on the system
    try:
        v = subprocess.run([PY311, "--version"], capture_output=True, text=True, check=True)
        print(f"System Python: {v.stdout.strip()}")
    except FileNotFoundError:
        raise RuntimeError(
            "python3.11 not found on PATH.\n"
            "Install it with:  sudo apt-get install python3.11 python3.11-venv\n"
            "Then re-run this cell."
        )

    if not VENV_PY.exists():
        print("Creating tf_env (Python 3.11) …")
        subprocess.check_call([PY311, "-m", "venv", str(VENV_DIR)])
        print(f"Created: {VENV_DIR}")
    else:
        print(f"Reusing existing: {VENV_DIR}")

    print("Installing packages into tf_env … (3-5 min on first run)")
    subprocess.check_call([str(VENV_PIP), "install", "--upgrade", "--quiet"] + PACKAGES)

    # Register as a Jupyter kernel so you can select it in the notebook UI
    subprocess.check_call([
        str(VENV_PY), "-m", "ipykernel", "install",
        "--user", "--name", "tf_env", "--display-name", "Python (tf_env)"
    ])
    print("\nDone. Switch the notebook kernel to 'Python (tf_env)' and continue from Section 2.")

> **Local only — action required after the cell above completes:**  
> Kernel → Change Kernel → *Python (tf_env)*  
> Then continue from **Section 2** onwards.  
>
> **Google Colab:** No kernel switch needed. Simply continue to the next cell after the install completes  
> (or restart the runtime if prompted for a TF version mismatch).

## 2  Imports & Configuration

In [ ]:
# ── CRITICAL: set before importing tensorflow or transformers ─────────────────
# TF 2.16+ ships Keras 3 by default. HuggingFace TF backend targets tf-keras.
import os, sys, json, shutil, warnings
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # force tf-keras, not Keras 3
warnings.filterwarnings("ignore")

from pathlib import Path
from typing  import List, Tuple, Dict

# ── environment detection ─────────────────────────────────────────────────────
IS_COLAB = "google.colab" in sys.modules

# ── data ──────────────────────────────────────────────────────────────────────
import numpy  as np
import pandas as pd

# ── TensorFlow + HuggingFace (TF backend) ─────────────────────────────────────
import tensorflow as tf
from transformers import (
    AutoTokenizer,
    TFAutoModelForSequenceClassification,
)

# ── tflite-support: schema objects + MetadataPopulator ────────────────────────
from tflite_support import metadata           as _tflite_meta
from tflite_support import metadata_schema_py_generated as _metadata_fb
import flatbuffers   # installed as a dependency of tflite-support; no separate install

# ── evaluation ────────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve,
)

# ── visualisation ─────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")     # headless-safe backend (works in Colab and local)
import matplotlib.pyplot as plt
import seaborn as sns
from   tqdm import tqdm

# ── version report ────────────────────────────────────────────────────────────
import transformers, sklearn, tflite_support
print(f"Python        : {sys.version}")
print(f"TensorFlow    : {tf.__version__}")
print(f"Transformers  : {transformers.__version__}")
print(f"Keras         : {tf.keras.__version__}")
print(f"tflite-support: {tflite_support.__version__}")
print(f"scikit-learn  : {sklearn.__version__}")
print(f"Colab env     : {IS_COLAB}")

In [ ]:
# ── reproducibility ───────────────────────────────────────────────────────────
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# ── GPU memory growth (prevents OOM on shared GPUs) ───────────────────────────
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print(f"GPUs available: {[g.name for g in gpus] or 'none (CPU)'}")

# ── paths — differ between local and Colab ────────────────────────────────────
if IS_COLAB:
    # In Colab, /content is the writable root.
    # Upload refined_training_data.csv via the Files panel (or mount Drive).
    # model.tflite + vocab.txt + labels.txt are written to /content/output/;
    # download them after training and copy to app/src/main/assets/ manually.
    ROOT       = Path("/content")
    ASSETS_DIR = ROOT / "output"
else:
    ROOT       = Path(".")
    ASSETS_DIR = ROOT / "app/src/main/assets"

DATASET_PATH       = ROOT / "refined_training_data.csv"
MODEL_OUT          = ASSETS_DIR / "model.tflite"
CHECKPOINT         = ROOT / "tf_env" / "mobilebert_tf_checkpoint"
TFLITE_UNQUANTIZED = ROOT / "tf_env" / "model_unquantized.tflite"
ASSETS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT.mkdir(parents=True, exist_ok=True)

# ── hyper-parameters ──────────────────────────────────────────────────────────
MODEL_NAME   = "google/mobilebert-uncased"
MAX_LEN      = 128        # practical sweet-spot for SMS; matches BertNLClassifier default
BATCH_SIZE   = 32         # power-of-2 batch; increase to 64 if GPU VRAM allows
EPOCHS       = 3
LR           = 2e-5
WARMUP_RATIO = 0.1
THRESHOLD    = 0.70       # must match SMSTransactionParser.kt isDebitScore < 0.70f

LABEL_NAMES  = ["Not Debit", "Debit"]

print(f"Dataset : {DATASET_PATH}")
print(f"Output  : {MODEL_OUT}")

# ── Colab: prompt to upload dataset if it doesn't exist ───────────────────────
if IS_COLAB and not DATASET_PATH.exists():
    from google.colab import files as _colab_files
    print("\n[Colab] refined_training_data.csv not found.")
    print("Upload it now via the prompt below, or mount Google Drive first.")
    uploaded = _colab_files.upload()
    import shutil as _shutil
    for fname in uploaded:
        _shutil.move(fname, str(DATASET_PATH))
    print(f"Saved to {DATASET_PATH}")

## 3  Dataset Loading & Exploratory Data Analysis

In [ ]:
df = pd.read_csv(DATASET_PATH)
print(f"Shape  : {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(5)

In [ ]:
print("=== Basic info ===")
print(df.info())
print("\n=== Null counts ===")
print(df.isnull().sum())
print("\n=== Duplicate rows ===")
print(df.duplicated().sum())

In [ ]:
# ── class-distribution plot ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Is_Debit distribution
counts = df["Is_Debit"].value_counts()
axes[0].bar(["Not Debit (0)", "Debit (1)"], counts.values,
            color=["#4CAF50", "#F44336"])
axes[0].set_title("Is_Debit Class Distribution")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f"{v:,}\n({v/len(df)*100:.1f}%)",
                 ha="center", fontsize=10)

# Is_Transaction distribution
counts_t = df["Is_Transaction"].value_counts()
axes[1].bar(["Non-txn (0)", "Transaction (1)"], counts_t.values,
            color=["#9E9E9E", "#2196F3"])
axes[1].set_title("Is_Transaction Distribution")
axes[1].set_ylabel("Count")
for i, v in enumerate(counts_t.values):
    axes[1].text(i, v + 200, f"{v:,}", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig("eda_class_distribution.png", dpi=120)
plt.show()
print("Saved: eda_class_distribution.png")

In [ ]:
# ── message-length distribution ───────────────────────────────────────────────
df["msg_len"] = df["Message"].str.len()

fig, ax = plt.subplots(figsize=(10, 4))
for label, color in [(0, "#4CAF50"), (1, "#F44336")]:
    subset = df[df["Is_Debit"] == label]["msg_len"]
    ax.hist(subset, bins=60, alpha=0.6, color=color,
            label=LABEL_NAMES[label])
ax.set_xlabel("Message length (chars)")
ax.set_ylabel("Count")
ax.set_title("Message Length Distribution by Class")
ax.legend()
ax.axvline(df["msg_len"].median(), color="black", linestyle="--",
           label=f"Median = {df['msg_len'].median():.0f}")
plt.tight_layout()
plt.savefig("eda_message_length.png", dpi=120)
plt.show()

print(df.groupby("Is_Debit")["msg_len"]
        .describe().round(0).to_string())

In [ ]:
# ── sample debit vs non-debit messages ───────────────────────────────────────
print("=== Sample DEBIT messages ===")
for msg in df[df["Is_Debit"]==1]["Message"].sample(3, random_state=SEED):
    print(f"  • {msg[:160]}")

print("\n=== Sample NON-DEBIT messages ===")
for msg in df[df["Is_Debit"]==0]["Message"].sample(3, random_state=SEED):
    print(f"  • {msg[:160]}")

## 4  Data Preprocessing

In [ ]:
# ── 4.1  clean ────────────────────────────────────────────────────────────────
df_clean = df.copy()

before = len(df_clean)
df_clean = df_clean.dropna(subset=["Message", "Is_Debit"])
print(f"Dropped {before - len(df_clean)} rows with NaN.")

before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["Message", "Is_Debit"])
print(f"Dropped {before - len(df_clean)} duplicate rows.")

df_clean["label"] = df_clean["Is_Debit"].astype(int)
df_clean["text"]  = df_clean["Message"].astype(str).str.strip()

print(f"\nFinal dataset size: {len(df_clean):,}")
print(df_clean["label"].value_counts().rename(index={0:"Not Debit", 1:"Debit"}))

In [ ]:
# ── 4.2  stratified train / val / test split  70 / 15 / 15 ───────────────────
X = df_clean["text"].tolist()
y = df_clean["label"].tolist()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)

print(f"Train : {len(X_train):>7,}  "
      f"(debit={sum(y_train):,}, {sum(y_train)/len(y_train)*100:.1f}%)")
print(f"Val   : {len(X_val):>7,}  "
      f"(debit={sum(y_val):,}, {sum(y_val)/len(y_val)*100:.1f}%)")
print(f"Test  : {len(X_test):>7,}  "
      f"(debit={sum(y_test):,}, {sum(y_test)/len(y_test)*100:.1f}%)")

## 5  Tokenisation & tf.data Pipeline

In [ ]:
print(f"Loading tokenizer: {MODEL_NAME} …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Vocab size: {tokenizer.vocab_size:,}")

In [ ]:
def tokenize_batch(texts: List[str]) -> dict:
    """Batch-tokenize with HuggingFace tokenizer; return numpy int32 arrays."""
    enc = tokenizer(
        texts,
        max_length     = MAX_LEN,
        padding        = "max_length",
        truncation     = True,
        return_tensors = "np",    # numpy – HuggingFace tokenizer is not TF-traceable
    )
    return {
        "input_ids"      : enc["input_ids"].astype(np.int32),
        "attention_mask" : enc["attention_mask"].astype(np.int32),
        "token_type_ids" : enc.get(
            "token_type_ids",
            np.zeros((len(texts), MAX_LEN), dtype=np.int32)
        ),
    }


def make_tf_dataset(
    texts     : List[str],
    labels    : List[int],
    batch_size: int,
    shuffle   : bool = False,
    seed      : int  = SEED,
) -> tf.data.Dataset:
    """
    Build a tf.data.Dataset that tokenises lazily (batch-level Python) and prefetches.

    Using from_generator so the HuggingFace tokenizer (not TF-traceable) runs in
    Python. Shuffle is applied by index-list permutation, not tf.data.shuffle,
    keeping the generator deterministic per epoch.
    """
    n       = len(texts)
    indices = list(range(n))

    def generator():
        idx_list = indices[:]
        if shuffle:
            rng = np.random.default_rng(seed)
            rng.shuffle(idx_list)
        for i in range(0, n, batch_size):
            batch_idx  = idx_list[i : i + batch_size]
            batch_text = [texts[j] for j in batch_idx]
            batch_lab  = [labels[j] for j in batch_idx]
            enc = tokenize_batch(batch_text)
            yield (
                {
                    "input_ids"      : enc["input_ids"],
                    "attention_mask" : enc["attention_mask"],
                    "token_type_ids" : enc["token_type_ids"],
                },
                np.array(batch_lab, dtype=np.int32),
            )

    output_signature = (
        {
            "input_ids"      : tf.TensorSpec(shape=(None, MAX_LEN), dtype=tf.int32),
            "attention_mask" : tf.TensorSpec(shape=(None, MAX_LEN), dtype=tf.int32),
            "token_type_ids" : tf.TensorSpec(shape=(None, MAX_LEN), dtype=tf.int32),
        },
        tf.TensorSpec(shape=(None,), dtype=tf.int32),
    )

    ds = tf.data.Dataset.from_generator(generator, output_signature=output_signature)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


print("Dataset factory defined.")

In [ ]:
train_ds = make_tf_dataset(X_train, y_train, BATCH_SIZE, shuffle=True)
val_ds   = make_tf_dataset(X_val,   y_val,   BATCH_SIZE, shuffle=False)
test_ds  = make_tf_dataset(X_test,  y_test,  BATCH_SIZE, shuffle=False)

steps_per_epoch = len(X_train) // BATCH_SIZE
val_steps       = len(X_val)   // BATCH_SIZE
test_steps      = len(X_test)  // BATCH_SIZE

print(f"Train batches/epoch : {steps_per_epoch:,}")
print(f"Val   batches       : {val_steps:,}")
print(f"Test  batches       : {test_steps:,}")

In [ ]:
# ── verify a single batch shape before training ───────────────────────────────
for (sample_inputs, sample_labels) in train_ds.take(1):
    for k, v in sample_inputs.items():
        print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
    print(f"  labels: shape={sample_labels.shape}, dtype={sample_labels.dtype}")

## 6  Model Setup

In [ ]:
print(f"Loading TF model: {MODEL_NAME} …")

# from_pt=False → load native TF weights (tf_model.h5) from HuggingFace Hub,
# avoiding cross-framework weight conversion.
hf_model = TFAutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels = 2,
    id2label   = {0: "0", 1: "1"},   # BertNLClassifier expects string labels "0"/"1"
    label2id   = {"0": 0, "1": 1},
    from_pt    = False,
)

total_params = sum(np.prod(v.shape) for v in hf_model.trainable_variables)
print(f"Trainable parameters: {total_params:,}")

In [ ]:
class TFMobileBERTClassifier(tf.keras.Model):
    """
    Thin wrapper around TFAutoModelForSequenceClassification.

    TFAutoModelForSequenceClassification.call() returns a TFSequenceClassifierOutput
    dataclass, not a plain tensor.  Keras .compile() + .fit() expect a plain tensor
    (or a dict of tensors).  This wrapper extracts .logits so the standard Keras
    loss and metric pipeline works correctly.
    """
    def __init__(self, hf_model):
        super().__init__()
        self.hf_model = hf_model

    def call(self, inputs, training=False):
        out = self.hf_model(inputs, training=training)
        return out.logits   # shape: (batch, num_labels)


keras_model = TFMobileBERTClassifier(hf_model)
print("TFMobileBERTClassifier wrapper created.")

In [ ]:
# ── linear warmup + linear decay LR schedule ─────────────────────────────────
# Replicates HuggingFace get_linear_schedule_with_warmup() used in the PyTorch notebook.

total_steps  = steps_per_epoch * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)


class LinearWarmupLinearDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    """Linear warmup 0→peak_lr over warmup_steps, then linear decay peak_lr→0."""

    def __init__(self, peak_lr: float, warmup_steps: int, total_steps: int):
        super().__init__()
        self.peak_lr      = float(peak_lr)
        self.warmup_steps = float(warmup_steps)
        self.total_steps  = float(total_steps)

    def __call__(self, step):
        step      = tf.cast(step, tf.float32)
        warmup_lr = self.peak_lr * (step / tf.maximum(self.warmup_steps, 1.0))
        progress  = (step - self.warmup_steps) / tf.maximum(
                     self.total_steps - self.warmup_steps, 1.0)
        decay_lr  = self.peak_lr * tf.maximum(0.0, 1.0 - progress)
        return tf.cond(step < self.warmup_steps,
                       lambda: warmup_lr, lambda: decay_lr)

    def get_config(self):
        return {
            "peak_lr"      : self.peak_lr,
            "warmup_steps" : int(self.warmup_steps),
            "total_steps"  : int(self.total_steps),
        }


lr_schedule = LinearWarmupLinearDecay(LR, warmup_steps, total_steps)
optimizer   = tf.keras.optimizers.Adam(
    learning_rate = lr_schedule,
    weight_decay  = 0.01,      # AdamW-style decoupled weight decay
    epsilon       = 1e-8,
    clipnorm      = 1.0,       # gradient clipping — mirrors clip_grad_norm_(1.0)
)

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
keras_model.compile(
    optimizer = optimizer,
    loss      = loss_fn,
    metrics   = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)

print(f"Total training steps : {total_steps:,}")
print(f"Warmup steps         : {warmup_steps:,}")
print("Model compiled.")

## 7  Training

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath          = str(CHECKPOINT / "best_weights"),
        save_weights_only = True,
        monitor           = "val_accuracy",
        save_best_only    = True,
        mode              = "max",
        verbose           = 1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor              = "val_accuracy",
        patience             = 2,
        mode                 = "max",
        verbose              = 1,
        restore_best_weights = True,   # reloads best weights automatically
    ),
]
print("Callbacks configured.")

In [ ]:
print("=" * 60)
print(f"  Training MobileBERT (TF) for {EPOCHS} epochs")
print("=" * 60)

history = keras_model.fit(
    train_ds,
    validation_data  = val_ds,
    epochs           = EPOCHS,
    steps_per_epoch  = steps_per_epoch,
    validation_steps = val_steps,
    callbacks        = callbacks,
    verbose          = 1,
)

print(f"\nBest val_accuracy: {max(history.history['val_accuracy']):.4f}")

In [ ]:
# ── training curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs_range = range(1, len(history.history["loss"]) + 1)

axes[0].plot(epochs_range, history.history["loss"],     "b-o", label="Train")
axes[0].plot(epochs_range, history.history["val_loss"], "r-o", label="Val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(epochs_range, history.history["accuracy"],     "b-o", label="Train")
axes[1].plot(epochs_range, history.history["val_accuracy"], "r-o", label="Val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.suptitle("MobileBERT (TF) Fine-tuning Curves", fontsize=13)
plt.tight_layout()
plt.savefig("training_curves_tf.png", dpi=120)
plt.show()
print("Saved: training_curves_tf.png")

## 8  Evaluation

`EarlyStopping(restore_best_weights=True)` already reloaded the best weights into `keras_model`,  
so no separate checkpoint reload step is needed.

In [ ]:
# ── collect test predictions ──────────────────────────────────────────────────
print("Running inference on test set …")
all_probs_list  = []
all_labels_list = []

for (batch_inputs, batch_labels) in tqdm(test_ds, total=test_steps, desc="Test"):
    logits = keras_model(batch_inputs, training=False)          # (batch, 2)
    probs  = tf.nn.softmax(logits, axis=-1).numpy()             # (batch, 2)
    all_probs_list.extend(probs[:, 1].tolist())                 # P(Is_Debit=1)
    all_labels_list.extend(batch_labels.numpy().tolist())

all_probs    = np.array(all_probs_list)
all_labels   = np.array(all_labels_list)
all_preds    = (all_probs >= 0.5).astype(int)                   # argmax equivalent
thresh_preds = (all_probs >= THRESHOLD).astype(int)             # app-parity threshold

In [ ]:
# ── core metrics ─────────────────────────────────────────────────────────────
def print_metrics(y_true, y_pred, probs, title=""):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    roc  = roc_auc_score(y_true, probs)

    print(f"\n{'='*50}")
    if title:
        print(f"  {title}")
    print(f"{'='*50}")
    print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"  ROC-AUC   : {roc:.4f}")
    print(f"{'='*50}")
    return {"accuracy": acc, "precision": prec, "recall": rec,
            "f1": f1, "roc_auc": roc}

metrics_argmax    = print_metrics(all_labels, all_preds,    all_probs,
                                  title="argmax predictions (threshold=0.5)")
metrics_threshold = print_metrics(all_labels, thresh_preds, all_probs,
                                  title=f"threshold={THRESHOLD} predictions (app parity)")

In [ ]:
# ── full classification report ────────────────────────────────────────────────
print("Classification Report (threshold parity):")
print(classification_report(all_labels, thresh_preds,
                             target_names=LABEL_NAMES, digits=4))

In [ ]:
# ── confusion matrix & ROC curve side by side ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- confusion matrix ---
cm = confusion_matrix(all_labels, thresh_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=axes[0])
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_title(f"Confusion Matrix (threshold={THRESHOLD})")

tn, fp, fn, tp = cm.ravel()
axes[0].set_xlabel(
    f"Predicted\n\nTN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}\n"
    f"Specificity={tn/(tn+fp):.3f}   Sensitivity={tp/(tp+fn):.3f}"
)

# --- ROC curve ---
fpr, tpr, _ = roc_curve(all_labels, all_probs)
auc_val     = roc_auc_score(all_labels, all_probs)
axes[1].plot(fpr, tpr, color="darkorange", lw=2,
             label=f"ROC (AUC = {auc_val:.4f})")
axes[1].plot([0, 1], [0, 1], color="navy", lw=1, linestyle="--")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("Receiver Operating Characteristic")
axes[1].legend(loc="lower right")

# mark operating threshold
_fpr, _tpr, _thresh = roc_curve(all_labels, all_probs)
idx = np.argmin(np.abs(_thresh - THRESHOLD))
axes[1].scatter(_fpr[idx], _tpr[idx], marker="*", s=200, color="red",
                label=f"App threshold={THRESHOLD}")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.savefig("evaluation_cm_roc_tf.png", dpi=120)
plt.show()
print("Saved: evaluation_cm_roc_tf.png")

In [ ]:
# ── probability-score distribution ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
for cls, color in [(0, "#4CAF50"), (1, "#F44336")]:
    probs_cls = [p for p, l in zip(all_probs, all_labels) if l == cls]
    ax.hist(probs_cls, bins=50, alpha=0.6, color=color, label=LABEL_NAMES[cls])
ax.axvline(THRESHOLD, color="black", linestyle="--",
           label=f"App threshold = {THRESHOLD}")
ax.set_xlabel("P(Is_Debit=1)")
ax.set_ylabel("Count")
ax.set_title("Predicted Probability Distribution by True Class")
ax.legend()
plt.tight_layout()
plt.savefig("evaluation_prob_dist_tf.png", dpi=120)
plt.show()
print("Saved: evaluation_prob_dist_tf.png")

## 9  TFLite Export (Native `tf.lite.TFLiteConverter`)

Converting from a `tf.function` concrete function with fixed input signatures is the most reliable  
approach for BERT-family models:  
- Tensor names are explicit and stable across TF versions  
- No ai-edge-torch or SavedModel dynamic-dispatch overhead  
- Dynamic-range quantization (weight-INT8, I/O float32) is compatible with `BertNLClassifier`

In [ ]:
# ── save tokenizer for later reloading (used in verification + copy to assets) ─
tokenizer.save_pretrained(str(CHECKPOINT))
print(f"Tokenizer saved to: {CHECKPOINT}")

# ── warmup call before tracing ─────────────────────────────────────────────────
# HuggingFace TF models create variables lazily on the first forward pass.
# Calling get_concrete_function() BEFORE a forward pass produces a graph with
# uninitialised variables → TFLite model outputs garbage.  This dummy call
# forces all variables to be created.
hf_model.trainable = False   # freeze for export
_dummy = {
    "input_ids"      : tf.zeros((1, MAX_LEN), dtype=tf.int32),
    "attention_mask" : tf.zeros((1, MAX_LEN), dtype=tf.int32),
    "token_type_ids" : tf.zeros((1, MAX_LEN), dtype=tf.int32),
}
_ = hf_model(_dummy, training=False)
print("Warmup pass complete — all variables initialised.")

In [ ]:
# ── define the serving function with explicit int32 input signatures ───────────
@tf.function(input_signature=[
    tf.TensorSpec([1, MAX_LEN], tf.int32, name="input_ids"),
    tf.TensorSpec([1, MAX_LEN], tf.int32, name="attention_mask"),
    tf.TensorSpec([1, MAX_LEN], tf.int32, name="token_type_ids"),
])
def serving_fn(input_ids, attention_mask, token_type_ids):
    out = hf_model(
        {
            "input_ids"      : input_ids,
            "attention_mask" : attention_mask,
            "token_type_ids" : token_type_ids,
        },
        training=False,
    )
    return tf.nn.softmax(out.logits, axis=-1)   # [1, 2] float32 scores


concrete_fn = serving_fn.get_concrete_function()
print("Concrete function traced.")
print(f"  Inputs  : {[t.name for t in concrete_fn.inputs[:3]]}")
print(f"  Outputs : {[t.name for t in concrete_fn.outputs]}")

In [ ]:
# ── convert to TFLite ─────────────────────────────────────────────────────────
converter = tf.lite.TFLiteConverter.from_concrete_functions(
    [concrete_fn], serving_fn
)
converter.optimizations         = [tf.lite.Optimize.DEFAULT]   # dynamic-range quantization
                                                                # weights→INT8, I/O →float32
                                                                # safe for BertNLClassifier
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,    # covers tf.einsum and other BERT ops
]
converter.allow_custom_ops       = False
converter._experimental_lower_tensor_list_ops = False

print("Converting to TFLite … (this may take 2–5 min)")
tflite_bytes = converter.convert()

TFLITE_UNQUANTIZED.write_bytes(tflite_bytes)
size_mb = len(tflite_bytes) / 1_048_576
print(f"TFLite model: {TFLITE_UNQUANTIZED}  ({size_mb:.1f} MB)")

# ── optional: print ops used (reveals if SELECT_TF_OPS are present) ───────────
tf.lite.experimental.Analyzer.analyze(model_path=str(TFLITE_UNQUANTIZED))

In [ ]:
# ── OPTIONAL: INT8 full quantization (commented out by default) ───────────────
# Full INT8 sets inference_input_type=tf.int8, which is INCOMPATIBLE with
# BertNLClassifier (expects float32 I/O).  Leave commented unless you are
# using a custom Android inference path.
#
# def representative_dataset():
#     for (batch_inputs, _) in train_ds.take(100):
#         yield [
#             tf.cast(batch_inputs["input_ids"][:1],       tf.float32),
#             tf.cast(batch_inputs["attention_mask"][:1],  tf.float32),
#             tf.cast(batch_inputs["token_type_ids"][:1],  tf.float32),
#         ]
#
# converter_int8 = tf.lite.TFLiteConverter.from_concrete_functions(
#     [concrete_fn], serving_fn
# )
# converter_int8.optimizations         = [tf.lite.Optimize.DEFAULT]
# converter_int8.representative_dataset = representative_dataset
# converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
# converter_int8.inference_input_type   = tf.int8
# converter_int8.inference_output_type  = tf.int8
# tflite_int8 = converter_int8.convert()
# (ROOT / "tf_env" / "model_int8.tflite").write_bytes(tflite_int8)
# print(f"INT8 model: {len(tflite_int8) / 1_048_576:.1f} MB")

print("(INT8 quantisation cell — uncomment above to enable)")

## 10  TFLite Metadata (BertNLClassifier compatibility)

`BertNLClassifier` in the Android TFLite Task Library reads embedded FlatBuffers metadata to locate:  
- the BERT WordPiece vocabulary (`vocab.txt`)  
- output label names (`labels.txt`)  

We use **`tflite-support`'s schema-generated Python objects** (`metadata_schema_py_generated`) to build  
the `ModelMetadata` FlatBuffer, then inject it with **`MetadataPopulator`**.  
This is the official tflite-support API — far cleaner than raw Builder calls.  
Available because we pinned to **Python 3.11** (tflite-support 0.4.4 has no Python 3.12 wheel).

In [ ]:
# ── 10.1  prepare associated files ────────────────────────────────────────────
VOCAB_SRC  = CHECKPOINT / "vocab.txt"
VOCAB_DST  = ROOT / "tf_env" / "vocab.txt"
LABELS_DST = ROOT / "tf_env" / "labels.txt"

shutil.copy(str(VOCAB_SRC), str(VOCAB_DST))

# BertNLClassifier maps output index → label string.
# SMSTransactionParser.kt checks: results.find { it.label == "1" }?.score
# Labels MUST be the literal strings "0" and "1", one per line.
LABELS_DST.write_text("0\n1\n")

print(f"vocab.txt  size : {VOCAB_DST.stat().st_size:,} bytes")
print(f"labels.txt      : {LABELS_DST.read_text()!r}")

In [ ]:
# ── 10.2  build metadata using tflite-support schema objects ──────────────────
#
# _metadata_fb.*T classes are generated Python bindings for metadata_schema.fbs.
# Each *T class maps directly to a FlatBuffers table, with Python attributes for
# every field.  Pack(builder) serialises to bytes.
# This replaces the brittle manual Builder.StartObject / PrependSlot approach.

def build_tflite_metadata(vocab_path: str, labels_path: str) -> bytes:
    """
    Build a TFLite ModelMetadata flatbuffer for BertNLClassifier using
    tflite-support schema-generated T objects.

    Schema reference:
    https://github.com/tensorflow/tflite-support/blob/master/
        tensorflow_lite_support/metadata/metadata_schema.fbs
    """
    # ── model-level metadata ──────────────────────────────────────────────────
    model_meta             = _metadata_fb.ModelMetadataT()
    model_meta.name        = "SMS Debit Classifier"
    model_meta.description = (
        "MobileBERT fine-tuned on Indian bank SMS. "
        "Binary classifier: index-1 = debit transaction."
    )
    model_meta.version     = "1.0.0"

    subgraph = _metadata_fb.SubGraphMetadataT()

    # ── input tensor metadata (3 tensors) ─────────────────────────────────────
    ids_md             = _metadata_fb.TensorMetadataT()
    ids_md.name        = "input_ids"
    ids_md.description = "Token IDs from BERT tokenizer."

    mask_md             = _metadata_fb.TensorMetadataT()
    mask_md.name        = "input_mask"
    mask_md.description = "Attention mask: 1 for real tokens, 0 for padding."

    seg_md             = _metadata_fb.TensorMetadataT()
    seg_md.name        = "segment_ids"
    seg_md.description = "Segment IDs (all zeros for single sentence)."

    subgraph.inputTensorMetadata = [ids_md, mask_md, seg_md]

    # ── BERT tokenizer process unit ───────────────────────────────────────────
    vocab_af             = _metadata_fb.AssociatedFileT()
    vocab_af.name        = os.path.basename(vocab_path)
    vocab_af.description = "Vocabulary file for the BERT WordPiece tokenizer."
    vocab_af.type        = _metadata_fb.AssociatedFileType.VOCABULARY

    bert_opts           = _metadata_fb.BertTokenizerOptionsT()
    bert_opts.vocabFile = [vocab_af]

    pu             = _metadata_fb.ProcessUnitT()
    pu.optionsType = _metadata_fb.ProcessUnitOptions.BertTokenizerOptions
    pu.options     = bert_opts

    subgraph.inputProcessUnits = [pu]

    # ── output tensor metadata ────────────────────────────────────────────────
    labels_af             = _metadata_fb.AssociatedFileT()
    labels_af.name        = os.path.basename(labels_path)
    labels_af.description = "Label file for classifier output."
    labels_af.type        = _metadata_fb.AssociatedFileType.TENSOR_AXIS_LABELS

    out_md                 = _metadata_fb.TensorMetadataT()
    out_md.name            = "output_scores"
    out_md.description     = "Softmax probabilities for each label."
    out_md.associatedFiles = [labels_af]

    subgraph.outputTensorMetadata = [out_md]
    model_meta.subgraphMetadata   = [subgraph]

    # ── serialise to FlatBuffer bytes ─────────────────────────────────────────
    b = flatbuffers.Builder(0)
    b.Finish(model_meta.Pack(b))
    return bytes(b.Output())


metadata_buf = build_tflite_metadata(str(VOCAB_DST), str(LABELS_DST))
print(f"Metadata buffer: {len(metadata_buf):,} bytes")

In [ ]:
# ── 10.3  inject metadata via MetadataPopulator ───────────────────────────────
#
# MetadataPopulator is the official tflite-support API for embedding metadata
# into a .tflite file.  It handles all FlatBuffer schema patching internally —
# no manual struct.pack, no TFLM sentinel, no raw buffer offsets.

model_buffer = bytearray(TFLITE_UNQUANTIZED.read_bytes())
populator    = _tflite_meta.MetadataPopulator.with_model_buffer(model_buffer)
populator.load_metadata_buffer(metadata_buf)
populator.load_associated_files([str(VOCAB_DST), str(LABELS_DST)])
populator.populate()

MODEL_OUT.write_bytes(bytes(populator.get_model_buffer()))
size_mb = MODEL_OUT.stat().st_size / 1_048_576
print(f"Metadata-injected model: {MODEL_OUT}  ({size_mb:.1f} MB)")

In [ ]:
# ── 10.4  copy associated files to assets/ ────────────────────────────────────
shutil.copy(str(VOCAB_DST),  str(ASSETS_DIR / "vocab.txt"))
shutil.copy(str(LABELS_DST), str(ASSETS_DIR / "labels.txt"))

print("Assets directory contents:")
for f in sorted(ASSETS_DIR.iterdir()):
    print(f"  {f.name:30s}  {f.stat().st_size / 1024:>8.1f} KB")

## 11  Model Verification

Run inference on the exported TFLite model using `tf.lite.Interpreter` to confirm  
the model file is valid before the Android build picks it up.

In [ ]:
# ── load TFLite interpreter and inspect tensor details ────────────────────────
interpreter = tf.lite.Interpreter(model_path=str(MODEL_OUT))
interpreter.allocate_tensors()

in_details  = interpreter.get_input_details()
out_details = interpreter.get_output_details()

print("Input tensors:")
for d in in_details:
    print(f"  [{d['index']}] {d['name']:45s} shape={d['shape']}  dtype={d['dtype'].__name__}")

print("\nOutput tensors:")
for d in out_details:
    print(f"  [{d['index']}] {d['name']:45s} shape={d['shape']}  dtype={d['dtype'].__name__}")

In [ ]:
# ── end-to-end inference smoke test ───────────────────────────────────────────
TEST_MESSAGES = [
    # expected debit = 1
    "Rs.95.15 on Zomato charged via Simpl. Food, groceries, commute, or medicines.",
    "Your A/c XX1234 debited INR 5,000.00 on 22-Feb-26 at ATM. Avl Bal: Rs.12,345.00",
    "INR 1299 debited from your HDFC Bank account for Netflix subscription.",
    # expected debit = 0
    "Your OTP for transaction is 583921. It is valid for 10 minutes. Do not share.",
    "Congratulations! You have won a free recharge of Rs.10. Click here to claim.",
    "Hi! Update your email id through WhatsApp or head to Vi App.",
]

# Reload tokenizer from the saved checkpoint
saved_tokenizer = AutoTokenizer.from_pretrained(str(CHECKPOINT))

print(f"{'SMS (truncated)':<65}  P(debit)  Decision (th={THRESHOLD})")
print("-" * 100)

for msg in TEST_MESSAGES:
    enc = saved_tokenizer(
        msg, max_length=MAX_LEN, padding="max_length",
        truncation=True, return_tensors="np"
    )

    # TFLite model expects int32 inputs (confirmed by input_details above)
    interpreter.set_tensor(in_details[0]["index"],
                           enc["input_ids"].astype(np.int32))
    interpreter.set_tensor(in_details[1]["index"],
                           enc["attention_mask"].astype(np.int32))
    if len(in_details) >= 3:
        interpreter.set_tensor(
            in_details[2]["index"],
            enc.get("token_type_ids",
                    np.zeros_like(enc["input_ids"])).astype(np.int32)
        )

    interpreter.invoke()
    scores   = interpreter.get_tensor(out_details[0]["index"])[0]   # [2]
    p_debit  = float(scores[1])
    decision = "DEBIT" if p_debit >= THRESHOLD else "not debit"

    print(f"{msg[:65]:<65}  {p_debit:.4f}    {decision}")

In [ ]:
# ── final summary ─────────────────────────────────────────────────────────────
model_size_mb = MODEL_OUT.stat().st_size / 1_048_576

print("=" * 60)
print("  TRAINING COMPLETE (TensorFlow Edition)")
print("=" * 60)
print(f"  Model       : {MODEL_NAME}")
print(f"  Epochs      : {EPOCHS}")
print(f"  Test acc    : {metrics_argmax['accuracy']*100:.2f}%")
print(f"  F1 (debit)  : {metrics_argmax['f1']:.4f}")
print(f"  ROC-AUC     : {metrics_argmax['roc_auc']:.4f}")
print(f"  TFLite size : {model_size_mb:.1f} MB")
print(f"  Output      : {MODEL_OUT}")
print("=" * 60)

## 12  Notes: Android Integration

### Files placed in `app/src/main/assets/`
| File | Purpose |
|------|--------|
| `model.tflite` | Fine-tuned MobileBERT TFLite model with embedded BertNLClassifier metadata |
| `vocab.txt` | BERT WordPiece vocabulary (30,522 tokens) |
| `labels.txt` | Class label mapping — `"0"` = Not Debit, `"1"` = Debit |

### No Android code changes required
`SMSTransactionParser.kt` calls `BertNLClassifier.createFromFileAndOptions(context, "model.tflite", options)`  
and checks `isDebitScore < 0.70f`.  The TF-trained model is a drop-in replacement:  
same file name, same label strings `"0"`/`"1"`, same float32 output scores.

### SELECT_TF_OPS — Gradle dependency check
The converter includes `tf.lite.OpsSet.SELECT_TF_OPS` to cover BERT ops (e.g. `tf.einsum`).  
Run `tf.lite.experimental.Analyzer.analyze()` after conversion (Section 9, Cell 3 output) to confirm  
whether SELECT ops are present.  If they are, add to `app/build.gradle.kts`:
```kotlin
implementation("org.tensorflow:tensorflow-lite-select-tf-ops:2.16.0")
```
The existing dependency `tensorflow-lite-task-text:0.4.4` already bundles many BERT ops,  
so SELECT ops may not be needed in practice.

### Google Colab: download output files
After training completes, download the three output files from `/content/output/`:
```python
from google.colab import files
files.download("/content/output/model.tflite")
files.download("/content/output/vocab.txt")
files.download("/content/output/labels.txt")
```
Then copy them to `app/src/main/assets/` in your local repo.

### Python version requirement
`tflite-support 0.4.4` requires **Python ≤ 3.11** (no 3.12 wheel).  
The local `tf_env/` venv is pinned to `python3.11`.  
Google Colab currently ships Python 3.10 or 3.11, both compatible.

### Re-training
Re-run from **Section 7** with a fresh kernel to skip environment setup.  
For a longer run: set `EPOCHS = 5`, `LR = 1e-5`.

### Coexistence with PyTorch notebook
This notebook writes `training_curves_tf.png`, `evaluation_cm_roc_tf.png`, `evaluation_prob_dist_tf.png`  
(distinct names from the PyTorch notebook's `training_curves.png` etc.) so both sets of charts  
can coexist in the project root for comparison.